In [2]:
from __future__ import annotations
from pathlib import Path
import sys
from typing import Dict

import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

# adjust this depending on where your notebook is relative to repo root
root = here if (here / "src").exists() else here.parents[1]
sys.path.insert(0, str(root / "src"))


# Detect Sundials (scikits.odes) availability
_HAS_SUNDIALS = False
try:
    from scikits.odes import ode as _scikits_ode  # type: ignore

    _HAS_SUNDIALS = True
except Exception:
    _HAS_SUNDIALS = False

# Import loader helpers
from chemistry.custom_network_loader import calculate_rate_coefficient

# External drivers that are NOT concentration variables (treated as constant drivers)
_EXTERNAL_DRIVERS = {"CR", "Photon", "CRP"}

In [ ]:
def prepare_initial_concentrations(species):
    """Prepare a default initial concentration vector for `species`.

    Returns an array-like matching earlier demo behavior.
    """
    if np is not None:
        x0 = np.zeros(len(species), dtype=float)
    else:
        x0 = [0.0] * len(species)

    for i, s in enumerate(species):
        if s == "e-":
            val = 1e-7
        else:
            val = 1e-8
        if np is not None:
            x0[i] = val
        else:
            x0[i] = val
    return x0

In [ ]:
def run_demo(
    rate_func,
    x0,
    species,
    network,
    t0=0.0,
    tf=1e3,
    T=100.0,
    av=10.0,
    count=10,
    S=None,
    reaction_data=None,
    cr_zeta=1.3e-17,
):
    """Run a short demo: sample evaluation and integration, printing results.

    This encapsulates the previous long __main__ demo block so the module's
    entrypoint remains concise.
    """
    # Evaluate rates and RHS at T
    rates, rhs = rate_func(x0, t=0.0, T=T, av=av)

    print("\nSample evaluation at T=100 K")
    print("First 10 reaction rates:")
    for j, r in enumerate(rates[:10]):
        print(f"  r{j+1:3d} = {r:.3e}")

    print("\nFirst 10 entries of dx/dt:")
    for i, val in enumerate(rhs[:10]):
        print(f"  d({species[i]})/dt = {val:.3e}")

    # Try SUNDIALS first, then fall back to SciPy
    if _HAS_SUNDIALS:
        try:
            print("\nRunning integration with SUNDIALS (CVODE)...")
            y0_use = x0
            if np is not None:
                y0_use = np.asarray(x0, dtype=float)

            sol = integrate_with_sundials(
                rate_func,
                y0_use,
                t0,
                tf,
                T=T,
                reaction_data=reaction_data,
                species=species,
                S=S,
                network=network,
                use_analytic_jac=True,
            )
            print("Integration finished with SUNDIALS")

            final = None
            try:
                final = sol.y[-1]
            except Exception:
                try:
                    final = sol.values.y[-1]
                except Exception:
                    final = None

            if final is not None:
                # enforce atom conservation and clamp
                from chemistry.util.cli_utils import (
                    print_clamped_table,
                    enforce_atom_conservation,
                )

                final = enforce_atom_conservation(final, y0_use, species, network)
                if np is not None:
                    final_clamped = np.maximum(final, 0.0)
                else:
                    final_clamped = [max(x, 0.0) for x in final]
                print_clamped_table(species, final, final_clamped, count=count)
            else:
                print("Solver returned result object; inspect `sol` for details.")
        except Exception as e:
            print(f"SUNDIALS integration failed: {e}")
            print("Falling back to SciPy if available...")
            # fall through to SciPy block

    # SciPy fallback or when SUNDIALS not available
    try:
        from scipy.integrate import solve_ivp
        from chemistry.util.jacobian_utils import analytic_jacobian

        print("Running integration with SciPy (solve_ivp)...")

        def rhs_scipy(t, y):
            return rate_func(y, t, T=T, av=av)[1]

        def jac_scipy(t, y):
            return analytic_jacobian(
                y, reaction_data, species, S, network, T=T, av=av, cr_zeta=cr_zeta
            )

        sol = solve_ivp(
            rhs_scipy, (t0, tf), x0, method="BDF", rtol=1e-6, atol=1e-12, jac=jac_scipy
        )
        final = sol.y[:, -1] if sol.y.ndim == 2 else sol.y[-1]
        from chemistry.util.cli_utils import (
            print_clamped_table,
            enforce_atom_conservation,
        )

        final = enforce_atom_conservation(final, x0, species, network)
        if np is not None:
            final_clamped = np.maximum(final, 0.0)
        else:
            final_clamped = [max(x, 0.0) for x in final]
        print_clamped_table(species, final, final_clamped, count=count)
    except Exception as e2:
        print(f"SciPy integration failed or not available: {e2}")


# ---------------------- Demo / CLI portion ----------------------
if __name__ == "__main__":

    # import helpers from cli_utils to keep __main__ concise
    from chemistry.util.cli_utils import print_clamped_table, enforce_atom_conservation

    # Load network from custom dir relative to this file
    curdir = Path(__file__).resolve().parent
    custom_dir = curdir / "custom"

    from chemistry.custom_network_loader import load_network

    network = load_network(custom_dir)
    if not network.get("species") or not network.get("reactions"):
        raise SystemExit("Failed to load network from custom directory")

    species, S, rate_func, reaction_data = make_ode_system(network)

    print("Built ODE system:")
    print(f"  species count = {len(species)}")
    print(f"  reaction count = {len(network['reactions'])}")

    # Prepare default initial concentrations and run the demo (sample + integrate)
    x0 = prepare_initial_concentrations(species)
    run_demo(
        rate_func,
        x0,
        species,
        network,
        t0=0.0,
        tf=1e3,
        T=100.0,
        av=10.0,
        count=10,
        S=S,
        reaction_data=reaction_data,
    )
